In [5]:
%pip install seleniumbase

  Using cached seleniumbase-4.47.3-py3-none-any.whl.metadata (88 kB)
  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached exceptiongroup-1.3.1-py3-none-any.whl.metadata (6.7 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached fasteners-0.20-py3-none-any.whl.metadata (4.8 kB)
  Using cached mycdp-1.3.6-py3-none-any.whl.metadata (3.7 kB)
  Using cached pynose-1.5.5-py3-none-any.whl.metadata (19 kB)
  Using cached platformdirs-4.9.4-py3-none-any.whl.metadata (4.7 kB)
  Using cached sbvirtualdisplay-1.4.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached parse-1.21.1-py2.py3-none-any.whl.metadata (21 kB)
  Using cached parse_type-0.6.6-py2.py3-none-any.whl.metadata (12 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached tabcompleter-1.4.0-py3-none-any.whl.metadat

In [ ]:
from seleniumbase import Driver
from bs4 import BeautifulSoup
import time
import csv
import re

def scrape_mobile_de_uc():
    # uc=True aktiviert den "Undetected ChromeDriver" Modus (Anti-Bot-Bypass)
    print("Starte getarnten Browser...")
    driver = Driver(uc=True, incognito=True)
    
    url = "https://suchen.mobile.de/fahrzeuge/search.html?dam=false&isSearchRequest=true&ms=17200%3B%3B11%3B&p=%3A35000&ref=quickSearch&s=Car&vc=Car"
    
    try:
        print(f"Lade Seite: {url}")
        driver.uc_open_with_reconnect(url, reconnect_time=5)
        
        print("\nWarte 15 Sekunden...")
        print("Falls du ein Captcha siehst oder die Seite immer noch lädt: LÖSE ES JETZT MANUELL!")
        time.sleep(15)
        
        # Cookie Banner wegklicken falls vorhanden (versucht es blind)
        try:
            driver.click('button:contains("Zustimmen")', timeout=2)
        except:
            pass
            
        # Bisschen scrollen für die Bilder/Lazy Loading
        driver.execute_script("window.scrollBy(0, 1000);")
        time.sleep(2)
        driver.execute_script("window.scrollBy(0, 1000);")
        time.sleep(2)
        
        # HTML extrahieren
        html = driver.page_source
        return html
        
    except Exception as e:
        print(f"Fehler: {e}")
        return None
    finally:
        driver.quit()

# --- Hauptprogramm ---
html_content = scrape_mobile_de_uc()

if html_content:
    soup = BeautifulSoup(html_content, 'html.parser')
    cars = []
    
    car_elements = soup.find_all('a', href=re.compile(r'/fahrzeuge/details\.html'))
    if not car_elements:
        car_elements = soup.find_all('div', {'data-testid': re.compile(r'result-listing|result-item', re.I)})

    for car in car_elements:
        title_elem = car.find(['h2', 'h3'])
        title = title_elem.text.strip() if title_elem else 'N/A'
        
        price_elem = car.find(lambda tag: tag.name in ['span', 'div'] and '€' in tag.text and len(tag.text) < 20)
        price = price_elem.text.strip() if price_elem else 'N/A'
        
        mileage = 'N/A'
        registration = 'N/A'
        
        for detail in car.find_all(['span', 'div', 'p']):
            text = detail.text.strip()
            if 'km' in text and len(text) < 15 and mileage == 'N/A':
                mileage = text
            if ('EZ' in text or re.search(r'\d{2}/\d{4}', text)) and len(text) < 20 and registration == 'N/A':
                registration = text

        if title != 'N/A' and price != 'N/A':
            cars.append({
                'Titel': title,
                'Preis': price,
                'Kilometerstand': mileage,
                'Erstzulassung': registration
            })

    unique_cars = [dict(t) for t in {tuple(d.items()) for d in cars}]

    if unique_cars:
        filename = 'mobile_de_seleniumbase.csv'
        with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
            fieldnames = ['Titel', 'Preis', 'Kilometerstand', 'Erstzulassung']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(unique_cars)
        print(f"\nBINGO! {len(unique_cars)} Fahrzeuge mit SeleniumBase in '{filename}' gespeichert!")
    else:
        print("\nFEHLER: Keine Autos im HTML gefunden. DataDome blockt uns womöglich immer noch.")
else:
    print("Scraping komplett fehlgeschlagen.")

Starte getarnten Browser...


*** chromedriver to download = 145.0.7632.117 (Previous Version)

https://storage.googleapis.com/chrome-for-testing-public/145.0.7632.117/mac-x64/chromedriver-mac-x64.zip ...
Download Complete!

Extracting ['chromedriver'] from chromedriver-mac-x64.zip ...
Unzip Complete!

The file [uc_driver] was saved to:
/opt/miniconda3/envs/Test/lib/python3.14/site-packages/seleniumbase/drivers/
uc_driver

Making [uc_driver 145.0.7632.117] executable ...
[uc_driver 145.0.7632.117] is now ready for use!

[Errno 86] Bad CPU type in executable: '/opt/miniconda3/envs/Test/lib/python3.14/site-packages/seleniumbase/drivers/uc_driver'


Exception: Missing a macOS dependency:
Your Mac needs Rosetta 2 to use UC Mode!
Run: "softwareupdate --install-rosetta"
Info: https://apple.stackexchange.com/a/408379/607628